In [1]:
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import xmltodict, json
import ast
import numbers
import shlex # package to construct the git command to subprocess format
import subprocess 
%matplotlib inline

In [2]:
def findModel(Parent,PathElements):
    for pe in PathElements:
        Parent = findNextChild(Parent,pe)
    return Parent

def findNextChild(Parent,ChildName):
    if len(Parent['Children']) >0:
        for child in range(len(Parent['Children'])):
            if Parent['Children'][child]['Name'] == ChildName:
                return Parent['Children'][child]
    else:
        return Parent[ChildName]
   
def replaceModel(Parent,modelPath,New):
    PathElements = modelPath.split('.')
    try:
        test = findModel(Parent,PathElements[:-1])[PathElements[-1]]
        findModel(Parent,PathElements[:-1])[PathElements[-1]] = New
    except:
        try:
            pos = 0
            for kid in findModel(Parent,PathElements[:-1])['Children']:
                if kid['Name'] == PathElements[-1]:
                    findModel(Parent,PathElements[:-1])['Children'][pos] = New
                    break
                pos +=1
        except:   
            print('Could not find parent node of model to over write for ' + modelPath)
            raise

In [94]:
# command= "git --git-dir=C:/GitHubRepos/ApsimX/.git --work-tree=C:/GitHubRepos/ApsimX checkout upstream/master C:/GitHubRepos/ApsimX/Tests/Validation/Wheat/Wheat.apsimx" 
# #command= "git --git-dir=C:/GitHubRepos/ApsimX/.git --work-tree=C:/GitHubRepos/ApsimX checkout C:/GitHubRepos/ApsimX/Models/Resources/Wheat.json" 
# comm=shlex.split(command) # This will convert the command into list format
# subprocess.run(comm, shell=True) # Run the git command

In [95]:
## Read wheat test file into json object
with open('C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\Wheat.apsimx','r') as WheatTestsJSON:
    WheatTests = json.load(WheatTestsJSON)
    WheatTestsJSON.close()
    ## read prototype wheat file into json object
with open('C:\GitHubRepos\ApsimX\Prototypes\WheatSimpleLeaf\WheatFewer.apsimx','r') as WheatPrototypeJSON:
    WheatPrototype = json.load(WheatPrototypeJSON)
    WheatPrototypeJSON.close()

In [96]:
#Copy prototype wheat model out of replacements and put it in replacements in test file
Replacements =  findModel(WheatPrototype,['Replacements'])
replaceModel(WheatTests,'Replacements',Replacements)

In [97]:
with open('C:\GitHubRepos\ApsimX\Prototypes\WheatSimpleLeaf\WheatSL.apsimx','w') as WheatTestsJSON:
    json.dump(WheatTests ,WheatTestsJSON,indent=2)

In [8]:
replacements = pd.read_excel('C:\GitHubRepos\ApsimX\Prototypes\WheatSimpleLeaf\SimpleLeafImplementation\VariableRenames.xlsx',index_col=0).to_dict()['SimpleLeaf']
with open(r'C:\GitHubRepos\ApsimX\Prototypes\WheatSimpleLeaf\WheatSL.apsimx', 'r') as file: 
    data = file.read() 
    for v in replacements.keys():
        data = data.replace(v, replacements[v])
        w = v.replace('Wheat','[Wheat]')
        rw = replacements[v].replace('Wheat','[Wheat]')
        data = data.replace(w, rw)
        
# Opening our text file in write only 
# mode to write the replaced content 
with open(r'C:\GitHubRepos\ApsimX\Prototypes\WheatSimpleLeaf\WheatSL.apsimx', 'w') as file: 
  
    # Writing the replaced data in our 
    # text file 
    file.write(data) 

In [16]:
from pathlib import Path
fileLoc = 'C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\data'
Allcols = []
pathlist = Path(fileLoc).glob('**/*.xlsx')
for path in pathlist:
    # because path is object not string
    obsDat = pd.read_excel(path, engine='openpyxl',sheet_name='Observed')
    newCols = []
    for c in obsDat.columns:
        if c=='Wheat.Phenology.PTQ':
            print(path)
        Allcols.append(c)
        if c in replacements.keys():
            newCols.append(c.replace(c,replacements[c]))
        else:
            newCols.append(c)
    obsDat.columns = newCols
    with pd.ExcelWriter(path, engine='openpyxl', mode='a',if_sheet_exists='replace') as writer: 
        workbook = writer.book
        obsDat.to_excel(writer,index=False,sheet_name='Observed')

In [12]:
for path in pathlist:
    # because path is object not string
    obsDat = pd.read_excel(path, engine='openpyxl',sheet_name='MaxLeafSize')
    newCols = []
    for c in obsDat.columns:
        if c in replacements.keys():
            newCols.append(c.replace(c,replacements[c]))
        else:
            newCols.append(c)
    obsDat.columns = newCols
    with pd.ExcelWriter(path, engine='openpyxl', mode='a',if_sheet_exists='replace') as writer: 
        workbook = writer.book
        obsDat.to_excel(writer,index=False,sheet_name='MaxLeafSize')

In [9]:
a = list(set(Allcols+list(replacements.keys())))
a.sort()

In [10]:
a

['([Wheat].Leaf.Transpiration + [Soil].SoilWater.Es + [MicroClimate].PrecipitationInterception)',
 'Clock.Today',
 'NDVIModel.Script.NDVI',
 'NDVIModel.Script.NDVI.se',
 'NDVIModel.Script.NDVIError',
 'Notes',
 'ObservedLayers.SW(1)',
 'ObservedLayers.SW(1)Error',
 'ObservedLayers.SW(10)',
 'ObservedLayers.SW(10)Error',
 'ObservedLayers.SW(2)',
 'ObservedLayers.SW(2)Error',
 'ObservedLayers.SW(3)',
 'ObservedLayers.SW(3)Error',
 'ObservedLayers.SW(4)',
 'ObservedLayers.SW(4)Error',
 'ObservedLayers.SW(5)',
 'ObservedLayers.SW(5)Error',
 'ObservedLayers.SW(6)',
 'ObservedLayers.SW(6)Error',
 'ObservedLayers.SW(7)',
 'ObservedLayers.SW(7)Error',
 'ObservedLayers.SW(8)',
 'ObservedLayers.SW(8)Error',
 'ObservedLayers.SW(9)',
 'ObservedLayers.SW(9)Error',
 'SimulationName',
 'Soil.Water.Volumetric(1)',
 'Soil.Water.Volumetric(10)',
 'Soil.Water.Volumetric(2)',
 'Soil.Water.Volumetric(3)',
 'Soil.Water.Volumetric(4)',
 'Soil.Water.Volumetric(5)',
 'Soil.Water.Volumetric(6)',
 'Soil.Water.Vo

In [16]:
obsDat

,SimulationName,Clock.Today,Notes,Wheat.Phenology.Zadok.Stage
0,FAR SAC W19-01MgmtHigh InputCvManning,2019-08-05,NaN,30.0
1,FAR SAC W19-01MgmtStandardCvManning,2019-08-05,NaN,30.0
2,FAR SAC W19-02FungicideFullCvManning,2019-08-05,NaN,30.0
3,FAR SAC W19-02FungicideNoneCvManning,2019-08-05,NaN,30.0
4,FAR SAC W19-01MgmtGrazedCvManning,2019-08-05,NaN,30.0
...,...,...,...,...
1192,FAR DMC W20-03MgmtStandardCvManning,2020-08-24,NaN,31.0
1193,FAR DMC W20-03MgmtGrazedCvManning,2020-08-24,NaN,31.0
1194,FAR DMC W20-03MgmtHigh InputCvManning,2020-10-20,NaN,65.0
1195,FAR DMC W20-03MgmtStandardCvManning,2020-10-20,NaN,65.0


In [139]:
test = pd.read_excel('C:/GitHubRepos/ApsimX/Tests/Validation/Wheat/Data/Observed.xlsx', engine='openpyxl',sheet_name='Observed')

In [140]:
test

,SimulationName,Clock.Today,Wheat.SowingDate,Wheat.SowingDate.DayOfYear,Wheat.SowingData.Cultivar,([Wheat].Leaf.Transpiration + [Soil].SoilWater.Es + [MicroClimate].PrecipitationInterception),sum([Soil].Water.MM),Soil.Water.Volumetric(1),Soil.Water.Volumetric(2),Soil.Water.Volumetric(3),...,Wheat.Spike.NonStructural.Wt,Wheat.Spike.Wt,Wheat.Spike.NConc,Wheat.Stem.NConc,Wheat.Stem.N,Wheat.Stem.NonStructural.Wt,Wheat.Stem.Wt,Wheat.Leaf.StemPopulation,Wheat.Phenology.HaunStage,Wheat.Phenology.FinalLeafNumber
0,APS14StubbleBareNRate000,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,APS14StubbleBareNRate040,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,APS14StubbleBareNRate080,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,APS14StubbleBareNRate200,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,APS14StubbleLucerneNRate000,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5362,Yucheng04,2005-06-07,NaT,NaN,NaN,NaN,378.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5363,Yucheng04,2005-06-08,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5364,Yucheng04,2005-06-11,NaT,NaN,NaN,NaN,383.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5365,Yucheng04,2005-06-17,NaT,NaN,NaN,NaN,378.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [142]:
cols = test.columns

In [144]:
replacements.keys()

dict_keys(['[Grain].NumberFunction.GrainNumber.GrainsPerGramOfStem', '[Phenology].Phyllochron.BasePhyllochron', '[Phenology].PhyllochronPpSensitivity', '[Leaf].Canopy.GreenExtinctionCoefficient', 'Wheat.Leaf.MainStemPopulation', 'sum(Soil.Water.MM)', 'sum(Soil.NO3.kgha)', 'Wheat.Grain.Nconc', 'Wheat.Leaf.CoverGreen', 'Wheat.Leaf.CoverTotal', 'Wheat.Leaf.Fn', 'Wheat.Leaf.Fw', 'Wheat.Leaf.Height', 'Wheat.Leaf.LAI', 'Wheat.Leaf.SpecificAreaCanopy', 'Wheat.Leaf.SpecificNitrogen', 'Wheat.Leaf.Transpiration', 'Wheat.Leaf.StemNumberPerPlant', 'Wheat.Leaf.StemPopulation', 'Wheat.Root.Depth', 'Wheat.Leaf.Tips', 'Wheat.Leaf.SpecificArea', 'Wheat.Phenology.HaunStage', 'Wheat.Phenology.PTQ', 'Wheat.Phenology.Phyllochron', 'Wheat.Arbitrator.DM.TotalFixationSupply', 'Wheat.Arbitrator.DM.TotalPlantDemand', 'Wheat.Stem.Live.StructuralWt', 'Wheat.Stem.Total.StructuralWt', 'Wheat.Stem.Live.StructuralN', 'Wheat.Stem.Total.StructuralN', 'Wheat.Stem.MinimumNConc', 'Wheat.Stem.CriticalNConc', 'Wheat.Stem.Ma

In [149]:
newCols = []
for c in cols:
    if c in replacements.keys():
        newCols.append(c.replace(c,replacements[c]))
    else:
        newCols.append(c)

In [150]:
newCols

['SimulationName',
 'Clock.Today',
 'Wheat.SowingDate',
 'Wheat.SowingDate.DayOfYear',
 'Wheat.SowingData.Cultivar',
 '([Wheat].Leaf.Transpiration + [Soil].SoilWater.Es + [MicroClimate].PrecipitationInterception)',
 'sum([Soil].Water.MM)',
 'Soil.Water.Volumetric(1)',
 'Soil.Water.Volumetric(2)',
 'Soil.Water.Volumetric(3)',
 'Soil.Water.Volumetric(4)',
 'Soil.Water.Volumetric(5)',
 'Soil.Water.Volumetric(6)',
 'Soil.Water.Volumetric(7)',
 'Soil.Water.Volumetric(8)',
 'Soil.Water.Volumetric(9)',
 'Soil.Water.Volumetric(10)',
 'Wheat.StemNumber',
 'Wheat.AboveGround.N',
 'Wheat.AboveGround.Wt',
 'Wheat.Ear.Wt',
 'Wheat.Ear.Nconc',
 'Wheat.Ear.N',
 'Wheat.Grain.NConc',
 'Wheat.Grain.Size',
 'Wheat.Grain.N',
 'Wheat.Grain.Number',
 'Wheat.Grain.Protein',
 'Wheat.Grain.Wt',
 'Wheat.Leaf.HaunStage',
 'Wheat.Leaf.Canopy.CoverTotal',
 'Wheat.Leaf.Canopy.CoverGreen',
 'Wheat.Leaf.Dead.NConc',
 'Wheat.Leaf.Dead.N',
 'Wheat.Leaf.Dead.Wt',
 'Wheat.Leaf.DeadCohortNo',
 'Wheat.Leaf.Ligules',
 'Whea

In [128]:
replacements

{'[Grain].NumberFunction.GrainNumber.GrainsPerGramOfStem': '[Grain].Number.GrainNumber.GrainsPerGramOfStem',
 '[Phenology].Phyllochron.BasePhyllochron': '[Leaf].Phyllochron.BasePhyllochron',
 '[Phenology].PhyllochronPpSensitivity': '[Leaf].PhyllochronPpSensitivity',
 '[Leaf].Canopy.GreenExtinctionCoefficient': '[Leaf].Canopy.GreenExtinctionCoefficient',
 'Wheat.Leaf.MainStemPopulation': 'Wheat.Population',
 'sum(Soil.Water.MM)': 'sum(Soil.Water.MM) as ProfileWater',
 'sum(Soil.NO3.kgha)': 'sum(Soil.NO3.kgha) as TotalNO3',
 'Wheat.Grain.Nconc': 'Wheat.Grain.NConc',
 'Wheat.Leaf.CoverGreen': 'Wheat.Leaf.Canopy.CoverGreen',
 'Wheat.Leaf.CoverTotal': 'Wheat.Leaf.Canopy.CoverTotal',
 'Wheat.Leaf.Fn': 'Wheat.Leaf.FN',
 'Wheat.Leaf.Fw': 'Wheat.Leaf.Canopy.FW',
 'Wheat.Leaf.Height': 'Wheat.Leaf.Canopy.Height',
 'Wheat.Leaf.LAI': 'Wheat.Leaf.Canopy.LAI',
 'Wheat.Leaf.SpecificAreaCanopy': 'Wheat.Leaf.Canopy.SpecificArea',
 'Wheat.Leaf.SpecificNitrogen': 'Wheat.Leaf.Canopy.SpecificNitrogen',
 'Wh

In [126]:
obsDat.co[x.replace() for x in obsDat.columns]

Index(['SimulationName', 'Clock.Today', 'Notes',
       'Wheat.Phenology.Zadok.Stage'],
      dtype='object')